# Backtesting d'une stratégie de trading — Momentum (croisement de moyennes mobiles)

## Portefeuille LVMH / TotalEnergies / Sanofi

Ce projet teste une stratégie de trading simple — un signal d'achat/vente basé sur le croisement de deux moyennes mobiles — sur trois valeurs du CAC 40 (LVMH, TotalEnergies, Sanofi), et compare sa performance à une stratégie passive "buy & hold" sur la même période.

## 1. Installation et imports

Installation de `yfinance` (récupération de données boursières) et des librairies de calcul standard.

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np

## 2. Récupération des données

Les tickers Yahoo Finance pour les valeurs du CAC 40 à Paris se terminent par `.PA` : LVMH (`MC.PA`), TotalEnergies (`TTE.PA`), Sanofi (`SAN.PA`). On récupère environ 6 ans de données quotidiennes, pour avoir assez d'historique pour tester des moyennes mobiles longues (50 et 200 jours) et traverser plusieurs cycles de marché, dont le krach Covid de 2020.

In [ ]:
tickers = ["MC.PA", "TTE.PA", "SAN.PA"]            # LVMH, TotalEnergies, Sanofi (suffixe .PA = Euronext Paris)
data = yf.download(tickers, start="2019-01-01", end="2026-09-27")  # téléchargement de l'historique (open/high/low/close/volume)
close_prices = data["Close"]                        # on ne garde que les prix de clôture, utilisés pour le calcul des moyennes mobiles
close_prices.head()                                 # aperçu des 5 premières lignes, pour vérifier que ça s'est bien chargé

## 3. Moyennes mobiles et signal de trading

La stratégie est un classique du trend-following : une moyenne mobile courte (50 jours) et une moyenne mobile longue (200 jours).

- Quand la MM50 passe **au-dessus** de la MM200 ("golden cross"), on considère la tendance haussière : on est **acheteur**.
- Quand la MM50 repasse **en-dessous** de la MM200 ("death cross"), on sort de la position.

Le signal est décalé d'un jour (`shift(1)`) : la décision prise à la clôture du jour J n'est appliquée qu'à partir du jour J+1, pour éviter un biais d'anticipation (on ne peut pas trader sur un prix de clôture qu'on ne connaît pas encore au moment où on déciderait).

In [ ]:
short_window = 50                                   # moyenne mobile courte : 50 jours de bourse (~2,5 mois)
long_window = 200                                   # moyenne mobile longue : 200 jours de bourse (~10 mois)

sma_short = close_prices.rolling(window=short_window).mean()  # moyenne glissante sur les 50 derniers jours, recalculée chaque jour
sma_long = close_prices.rolling(window=long_window).mean()    # idem sur 200 jours

signal = (sma_short > sma_long).astype(int)         # 1 si la MM50 est au-dessus de la MM200 (tendance haussière), 0 sinon
position = signal.shift(1)                          # on décale d'un jour : on trade le lendemain du signal, jamais le jour même

position.tail()                                      # aperçu des dernières positions calculées (0 ou 1, par action)

## 4. Rendements de la stratégie vs buy & hold

On calcule d'abord le rendement journalier de chaque action avec `pct_change()`, c'est-à-dire `(Prix_jour - Prix_veille) / Prix_veille`. C'est la même logique que dans le projet VaR : on raisonne en rendements (en %) plutôt qu'en variations de prix en euros, pour pouvoir comparer et agréger des actions qui n'ont pas du tout le même niveau de prix (LVMH vaut plusieurs centaines d'euros, Sanofi une centaine, TotalEnergies une soixantaine).

Pour obtenir le rendement **de la stratégie**, on multiplie le rendement du jour par la position prise la veille (`position`, qui vaut 0 ou 1) :

- si `position = 1` (on est investi, golden cross actif) → on encaisse le rendement du jour, à l'identique du buy & hold
- si `position = 0` (on est sorti, death cross actif) → le rendement de la stratégie ce jour-là est nul : on est en cash, on ne gagne ni ne perd (simplification — en réalité le cash est généralement placé à un taux sans risque, mais on l'ignore ici pour rester sur l'essentiel)

En multipliant jour après jour les `(1 + rendement)` entre eux (`cumprod()`), on reconstitue l'évolution du capital, en partant d'une base 1 (= 1 euro investi au premier jour).

In [ ]:
returns = close_prices.pct_change()                 # rendement journalier = (prix du jour - prix de la veille) / prix de la veille

strategy_returns = position * returns               # rendement encaissé seulement les jours où position = 1 (sinon 0, on est en cash)
buy_hold_returns = returns                           # la référence passive : toujours investi, sans aucun signal

strategy_cumulative = (1 + strategy_returns).cumprod()   # capital cumulé de la stratégie, base 1 au départ
buy_hold_cumulative = (1 + buy_hold_returns).cumprod()   # capital cumulé du buy & hold, base 1 au départ

strategy_cumulative.tail()                           # aperçu des dernières valeurs du capital stratégie, par action

## 5. Visualisation : stratégie vs buy & hold

On trace, pour chaque action, l'évolution du capital selon les deux approches. C'est la façon la plus parlante de juger une stratégie de trading : au-delà des chiffres, on voit visuellement si le filtre moyennes mobiles a permis d'éviter les grosses baisses (comme le krach Covid de mars 2020), ou s'il est resté à la traîne pendant les phases de hausse continue (où rester investi en permanence, sans signal, est optimal par construction).

On utilise une échelle **logarithmique** sur l'axe des Y : sur plusieurs années, un graphique en échelle normale écraserait les variations du début de période face aux niveaux de fin de période. En échelle log, une même variation en % occupe toujours le même espace visuel, quelle que soit l'époque.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 12))          # une rangée de graphique par action (3 actions = 3 sous-graphiques empilés)

for i, ticker in enumerate(tickers):                        # on boucle sur les 3 tickers pour remplir chaque sous-graphique
    ax = axes[i]
    ax.plot(strategy_cumulative.index, strategy_cumulative[ticker], label="Stratégie MM50/MM200", color="steelblue")
    ax.plot(buy_hold_cumulative.index, buy_hold_cumulative[ticker], label="Buy & Hold", color="gray", linestyle="--")
    ax.set_yscale("log")                                     # échelle log, pour les raisons expliquées ci-dessus
    ax.set_title(f"{ticker} — capital cumulé (base 1)")
    ax.legend()

plt.tight_layout()                                           # évite que les titres/légendes de chaque sous-graphique se chevauchent
plt.savefig("capital_cumule_par_action.png", dpi=150, bbox_inches="tight")
plt.show()

Sur LVMH, la stratégie est nettement en-dessous du buy & hold sur toute la période : c'est le défaut classique du trend-following sur une valeur qui monte de façon quasi continue avec peu de vrais retournements — chaque aller-retour (sortie puis rentrée) coûte un peu de performance par rapport à "rester investi sans rien faire", et les phases plates (en cash, visibles comme des segments horizontaux — la position est à 0, donc le rendement de la stratégie est nul et le capital ne bouge plus) font manquer une partie de la hausse.

## 6. Métriques de performance

Le graphique donne une impression visuelle, mais pour comparer rigoureusement deux stratégies il faut des chiffres. On calcule quatre indicateurs classiques en gestion de portefeuille :

- **CAGR** (*Compound Annual Growth Rate*) : le rendement annuel moyen, en tenant compte des intérêts composés. C'est la question "si cette performance avait été constante chaque année, à quel taux annuel ça correspondrait-il ?"
- **Volatilité annualisée** : l'écart-type des rendements journaliers, ramené à une échelle annuelle (on multiplie par √252, le nombre de jours de bourse dans une année — c'est la même logique de racine carrée du temps qu'on utilise pour transposer une VaR 1 jour en VaR 10 jours).
- **Ratio de Sharpe** : le rendement annualisé rapporté à la volatilité annualisée. Il répond à la question "est-ce que la performance obtenue compense le risque pris ?" — à rendement égal, la stratégie la moins volatile a le meilleur Sharpe. Ici on suppose un taux sans risque nul, pour simplifier.
- **Max Drawdown** : la pire perte subie depuis un plus haut historique, à n'importe quel moment de la période. C'est l'indicateur qui parle le plus concrètement à un investisseur ("combien aurais-je pu perdre si j'étais entré au pire moment ?").

On calcule ces 4 indicateurs séparément pour la stratégie et pour le buy & hold, sur chacune des 3 actions.

In [ ]:
def compute_metrics(returns):
    returns = returns.dropna()                          # on retire les jours sans valeur (avant que la MM200 existe, ex.)
    cumulative = (1 + returns).cumprod()                 # capital cumulé recalculé sur la série nettoyée

    n_years = len(returns) / 252                         # nombre d'années, à raison de ~252 jours de bourse par an
    cagr = cumulative.iloc[-1] ** (1 / n_years) - 1       # rendement annualisé composé

    vol_annual = returns.std() * np.sqrt(252)             # volatilité annualisée

    sharpe = (returns.mean() * 252) / vol_annual          # rendement annualisé / volatilité annualisée (rf = 0)

    running_max = cumulative.cummax()                     # plus haut niveau de capital atteint à chaque instant
    drawdown = cumulative / running_max - 1                # perte en % par rapport à ce plus haut
    max_drawdown = drawdown.min()                          # la pire perte depuis un sommet, sur toute la période

    return pd.Series({
        "CAGR": cagr,
        "Vol annualisée": vol_annual,
        "Sharpe": sharpe,
        "Max Drawdown": max_drawdown,
    })

summary_rows = {}
for ticker in tickers:                                     # on répète le calcul pour chaque action
    summary_rows[f"{ticker} — Stratégie"] = compute_metrics(strategy_returns[ticker])
    summary_rows[f"{ticker} — Buy & Hold"] = compute_metrics(buy_hold_returns[ticker])

summary = pd.DataFrame(summary_rows).T                      # on assemble les résultats dans un seul tableau

summary.style.format({                                      # mise en forme en %/ratio pour la lecture
    "CAGR": "{:.2%}",
    "Vol annualisée": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Max Drawdown": "{:.2%}",
})

**Lecture du tableau** : seule TotalEnergies tient la promesse du trend-following (Sharpe légèrement meilleur, drawdown divisé par deux). Sur LVMH, la stratégie est clairement ratée : CAGR négatif, Sharpe quasi nul, et un drawdown **plus profond** que le buy & hold — alors qu'elle est censée protéger des grosses baisses. Le krach Covid de mars 2020 explique ce cas : une chute aussi rapide est mal gérée par une moyenne mobile 200 jours, qui réagit toujours avec retard (voir la section 8).

## 7. Construction du portefeuille

Comme pour le projet VaR, on pondère les trois actions à parts égales (1/3 chacune). C'est une simplification qui suppose un rééquilibrage quotidien parfait à 1/3-1/3-1/3 : en pratique les poids dériveraient avec la performance relative de chaque action, mais ce choix permet de se concentrer sur l'effet de la stratégie plutôt que sur un problème d'allocation.

Pour obtenir le rendement du portefeuille un jour donné, on fait la moyenne pondérée des rendements des 3 actions ce jour-là (`.dot(weights)` = produit scalaire ligne par ligne). On refait ça séparément pour la version "stratégie" et la version "buy & hold", puis on recalcule le capital cumulé de chaque portefeuille.

In [ ]:
weights = np.array([1/3, 1/3, 1/3])                                  # pondération égale entre LVMH, TotalEnergies, Sanofi

portfolio_strategy_returns = strategy_returns.dot(weights).dropna()   # rendement journalier du portefeuille "stratégie"
portfolio_buy_hold_returns = buy_hold_returns.dot(weights).dropna()   # rendement journalier du portefeuille "buy & hold"

portfolio_strategy_cumulative = (1 + portfolio_strategy_returns).cumprod()   # capital cumulé du portefeuille stratégie
portfolio_buy_hold_cumulative = (1 + portfolio_buy_hold_returns).cumprod()   # capital cumulé du portefeuille buy & hold

portfolio_strategy_cumulative.tail()

In [ ]:
portfolio_summary = pd.DataFrame({                                    # on réutilise la fonction compute_metrics définie plus haut
    "Stratégie": compute_metrics(portfolio_strategy_returns),
    "Buy & Hold": compute_metrics(portfolio_buy_hold_returns),
}).T

display(portfolio_summary.style.format({
    "CAGR": "{:.2%}", "Vol annualisée": "{:.2%}", "Sharpe": "{:.2f}", "Max Drawdown": "{:.2%}",
}))

plt.figure(figsize=(12, 6))
plt.plot(portfolio_strategy_cumulative.index, portfolio_strategy_cumulative, label="Portefeuille — Stratégie MM50/MM200", color="steelblue")
plt.plot(portfolio_buy_hold_cumulative.index, portfolio_buy_hold_cumulative, label="Portefeuille — Buy & Hold", color="gray", linestyle="--")
plt.yscale("log")                                                      # même raison qu'avant : lisibilité sur plusieurs années
plt.title("Portefeuille équipondéré LVMH / TotalEnergies / Sanofi — capital cumulé")
plt.legend()
plt.savefig("capital_cumule_portefeuille.png", dpi=150, bbox_inches="tight")
plt.show()

**Résultat net** : sur le portefeuille consolidé, le buy & hold bat la stratégie sur toute la ligne — CAGR (10,15% vs 3,38%) **et** Sharpe (0,59 vs 0,32), même si la stratégie réduit bien le drawdown max (-26% vs -37%) et la volatilité (12,9% vs 19,5%).

C'est l'inverse du résultat du papier de référence de Faber (2007) sur l'indice S&P 500 global, où la même règle améliore le Sharpe. L'explication : sur un indice très diversifié, les faux signaux (whipsaws) de chaque action individuelle ont tendance à se moyenner, et les crises y sont souvent plus lentes à se former ; sur un panier de seulement 3 actions, un épisode rapide comme le krach Covid suffit à pénaliser durablement la performance, sans se compenser statistiquement avec seulement 2 autres lignes.

## 8. Zoom sur le krach Covid (le cas du whipsaw)

Pour illustrer concrètement le problème identifié plus haut, on isole la période novembre 2019 - décembre 2020 et on rebase les deux courbes à 1 au début de cette fenêtre (pour comparer uniquement l'évolution *pendant* cet épisode, indépendamment du niveau de capital accumulé avant).

On ajoute un second graphique en dessous : la **part du portefeuille investie** à chaque instant (`position.mean(axis=1)`, la moyenne des 3 signaux 0/1 — 1 = les 3 actions investies, 0 = les 3 en cash, 0.33/0.67 = une partie seulement). Ça permet de voir précisément *quand* la stratégie est sortie et rentrée, et de comparer ce timing avec la chute et le rebond réels du marché au-dessus.

In [ ]:
zoom_start = "2019-11-01"
zoom_end = "2020-12-31"

strat_zoom = portfolio_strategy_cumulative.loc[zoom_start:zoom_end]       # on découpe la série sur la fenêtre choisie
bh_zoom = portfolio_buy_hold_cumulative.loc[zoom_start:zoom_end]

strat_zoom_rebased = strat_zoom / strat_zoom.iloc[0]                       # on repart de 1 au début de la fenêtre (comparaison isolée)
bh_zoom_rebased = bh_zoom / bh_zoom.iloc[0]

exposure = position.mean(axis=1).loc[zoom_start:zoom_end]                  # part du portefeuille investie à chaque instant

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})  # 2 sous-graphiques empilés

ax1.plot(strat_zoom_rebased.index, strat_zoom_rebased, label="Stratégie MM50/MM200", color="steelblue")
ax1.plot(bh_zoom_rebased.index, bh_zoom_rebased, label="Buy & Hold", color="gray", linestyle="--")
ax1.set_title("Zoom sur le krach Covid — capital rebasé à 1 au 01/11/2019")
ax1.legend()

ax2.fill_between(exposure.index, exposure, step="post", color="steelblue", alpha=0.4)  # aire en escalier = part investie
ax2.set_ylabel("Part investie")
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("zoom_covid.png", dpi=150, bbox_inches="tight")
plt.show()

On voit nettement le décalage : le marché touche son plus bas vers la mi-mars 2020, mais la "part investie" ne redescend à son minimum qu'après coup, et surtout elle ne remonte pleinement qu'après une bonne partie du rebond — la stratégie encaisse une portion du risque baissier et manque une portion de la hausse qui suit. C'est exactement le mécanisme qui explique le sous-rendement observé plus haut.

## Conclusion

| | CAGR Stratégie | CAGR B&H | Sharpe Stratégie | Sharpe B&H | Max DD Stratégie | Max DD B&H |
|---|---|---|---|---|---|---|
| LVMH | -2.10% | 7.66% | 0.00 | 0.40 | -60.40% | -52.61% |
| TotalEnergies | 10.22% | 13.53% | 0.60 | 0.59 | -27.04% | -56.79% |
| Sanofi | -0.71% | 3.63% | 0.05 | 0.27 | -29.83% | -27.80% |
| **Portefeuille** | **3.38%** | **10.15%** | **0.32** | **0.59** | **-26.29%** | **-37.17%** |

**Résultat** : sur ce panier de 3 valeurs, la stratégie de croisement de moyennes mobiles (50/200 jours) **ne bat pas** le buy & hold, ni en rendement ni en rendement ajusté du risque (Sharpe 0,32 contre 0,59). Elle réduit nettement la volatilité et le drawdown max, mais au prix d'un rendement sacrifié trop important pour compenser. Seule TotalEnergies fait exception, avec un Sharpe légèrement meilleur et un drawdown divisé par deux.

Le zoom sur le krach Covid explique en grande partie ce résultat : la moyenne mobile 200 jours réagit avec retard à une chute rapide, et la stratégie sort du marché après le plus gros de la baisse puis rentre après une partie du rebond — elle cumule une partie du risque baissier et manque une partie de la hausse qui suit.

**Limites et pistes d'amélioration** :
- Seulement 3 actions, non diversifiées sectoriellement (LVMH et Sanofi sont assez corrélées à la conjoncture mondiale) — un panier plus large ou un indice lisserait les faux signaux individuels
- Le cash (position = 0) est supposé rémunéré à 0%, alors qu'un taux sans risque réel aurait légèrement amélioré la performance de la stratégie
- Aucun frais de transaction n'est modélisé ici, contrairement au projet VaR où un coût par aller-retour avait été intégré — à ajouter pour une version plus réaliste
- Les paramètres (50/200 jours) viennent de la littérature et n'ont pas été optimisés sur ces 3 actions précises : c'est volontaire (éviter le sur-ajustement), mais ça n'exclut pas qu'un autre couple de fenêtres, ou un filtre de confirmation (ex: n'agir qu'après 2-3 jours de croisement confirmé), eût donné un résultat différent sur cette période

In [ ]:
close_prices.to_csv("prix_actions.csv")                 # sauvegarde des prix bruts, pour reproductibilité
summary.to_csv("metriques_par_action.csv")              # tableau des métriques par action
portfolio_summary.to_csv("metriques_portefeuille.csv")  # tableau des métriques du portefeuille